# AI in Space — Space Weather Launch Safety Predictor
## Educational Laboratory Notebook

This notebook walks through all 9 required tasks for the IBM AI in Space laboratory.

---
> **Educational Disclaimer:** This is an educational risk-assessment system.
> The risk scores and recommendations are derived from a simplified educational model.
> This is NOT an operational NASA launch-safety or space-weather forecasting system.

## Task 1 — Install Required Libraries

Install and verify all required dependencies.

In [ ]:
import sys
import subprocess

required_packages = [
    'pandas',
    'numpy',
    'scikit-learn',
    'matplotlib',
    'streamlit',
    'joblib',
    'requests'
]

print('Verifying required libraries...')
print('-' * 40)
all_ok = True

import_names = {
    'scikit-learn': 'sklearn',
}

for pkg in required_packages:
    import_name = import_names.get(pkg, pkg)
    try:
        mod = __import__(import_name)
        version = getattr(mod, '__version__', 'installed')
        print(f'  ✓  {pkg:<20} {version}')
    except ImportError:
        print(f'  ✗  {pkg:<20} NOT FOUND')
        all_ok = False

print('-' * 40)
if all_ok:
    print('All libraries verified.')
else:
    print('Some libraries are missing. Run: pip install -r requirements.txt')

## Task 2 — Download and Load Dataset

Load `data/space_weather_unified.csv` and display basic information.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import load_dataset

df = load_dataset()
print(f'\nDataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

## Task 3 — Prepare Data

Clean the raw dataset:
- Fill missing values
- Remove duplicates
- Parse timestamps
- Extract flare class and magnitude
- Calculate event duration

In [ ]:
from src.data_cleaning import clean_space_weather_data

space_df = clean_space_weather_data(df)

print('\nCleaned dataset info:')
print(space_df[['event_type', 'flare_class', 'flare_magnitude', 'duration_minutes']].head(10))

## Task 4 — Explore and Analyze Data

Analyze:
- Event type distribution
- Temporal patterns
- Solar flare characteristics
- Geomagnetic storm statistics

In [ ]:
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

from src.eda import run_eda
eda_stats = run_eda(space_df)

## Task 5 — Feature Engineering

Build historical 48-hour risk features for each date.

**Data leakage protection:** For each prediction date, only events
from the previous 48 hours (not including the current date) are used.

In [ ]:
from src.feature_engineering import build_risk_features

risk_features_df = build_risk_features(space_df)

print('\nFeature matrix shape:', risk_features_df.shape)
print('\nSample features:')
risk_features_df.head(5)

## Task 6 — Risk Scoring

Calculate a 0–100 educational launch risk score using the formula:

```
x_score     = min(xclass_flare_count × 40, 40)
m_score     = min(mclass_flare_count × 25, 25)
kp_score    = (max_kp_index / 9) × 20
trend_score = min(max((event_trend - 1) × 15, 0), 15)
total       = min(x_score + m_score + kp_score + trend_score, 100)
```

Risk levels:
- LOW (0–20) → GO
- MODERATE (20–40) → CAUTION
- HIGH (40–60) → DELAY
- EXTREME (≥60) → NO-GO

In [ ]:
from src.risk_scoring import apply_risk_scoring

scored_df = apply_risk_scoring(risk_features_df)

print('\nTop 10 highest-risk dates:')
scored_df.nlargest(10, 'risk_score')[['date', 'risk_score', 'risk_level', 'recommendation']]

## Task 7 — Train and Evaluate Decision Model

Train a Random Forest classifier with a **time-based split**.
Data before 2025-01-01 is used for training; data on or after is used for testing.
The data is never shuffled to prevent look-ahead bias.

In [ ]:
from src.model_training import train_model, FEATURE_COLS
from src.model_evaluation import evaluate_model

model, X_train, X_test, y_train, y_test, y_pred = train_model(scored_df)
results = evaluate_model(model, X_train, X_test, y_train, y_test, y_pred, FEATURE_COLS)

print('\nFeature importances:')
print(results['feature_importances'])

## Task 8 — Save Model and Risk Data

Save:
- `models/launch_decision_model.pkl` — trained Random Forest
- `models/space_weather_data.pkl` — current stats, feature cols, recent features

In [ ]:
from src.persistence import save_artifacts

save_artifacts(model, scored_df, FEATURE_COLS)
print('\nModel and data saved successfully.')

## Task 9 — Date-Range Go/No-Go Dashboard

Analyze a date range and produce the three required charts:
1. Risk Score per Day
2. Daily Recommendation (GO / CAUTION / DELAY / NO-GO)
3. Solar Events in 48-Hour Window

**For the full interactive dashboard:**
```bash
streamlit run dashboard/app.py
```

In [ ]:
import pandas as pd
%matplotlib inline

from src.prediction import load_saved_data
from dashboard.dashboard_utils import filter_date_range, compute_date_range_summary
from dashboard.charts import (
    chart_risk_score_per_day,
    chart_daily_recommendation,
    chart_solar_events_48h,
)

data = load_saved_data()
full_df = data['recent_features'].copy()
full_df['date'] = pd.to_datetime(full_df['date'])

# Select date range (adjust as needed)
START_DATE = full_df['date'].min().strftime('%Y-%m-%d')
END_DATE   = full_df['date'].max().strftime('%Y-%m-%d')

print(f'Analyzing {START_DATE} → {END_DATE}')

filtered = filter_date_range(full_df, START_DATE, END_DATE)
summary  = compute_date_range_summary(filtered)

print(f'Total days   : {summary["total_days"]}')
print(f'Avg score    : {summary["avg_risk_score"]}')
print(f'GO           : {summary["go_days"]}')
print(f'CAUTION      : {summary["caution_days"]}')
print(f'DELAY        : {summary["delay_days"]}')
print(f'NO-GO        : {summary["no_go_days"]}')
print(f'Recommendation: {summary["overall_recommendation"]}')

import matplotlib.pyplot as plt

fig1 = chart_risk_score_per_day(filtered)
plt.show()

fig2 = chart_daily_recommendation(filtered)
plt.show()

fig3 = chart_solar_events_48h(filtered)
plt.show()